# CommGuard benign corpus v2

Restore one exact prior-session calibration as input evidence, then run a fresh current-session calibration as the actual collection gate. The two artifacts are never interchangeable. A bounded pilot is the default and the standard 24-run corpus is an explicit opt-in.


In [ ]:
import importlib
import os
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_benign_corpus_v2"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
REVIEWED_COMMIT = ""  # Required: immutable 40-character commit visible on origin.
REPOSITORY = Path("/kaggle/working/commguard-source")

if not re.fullmatch(r"[0-9a-f]{40}", REVIEWED_COMMIT):
    raise RuntimeError("Set REVIEWED_COMMIT to the reviewed, pushed 40-character commit SHA.")
if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )
if not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "origin", REVIEWED_COMMIT], check=True)
subprocess.run(
    ["git", "-C", str(REPOSITORY), "checkout", "--detach", REVIEWED_COMMIT], check=True
)
head = subprocess.run(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(REPOSITORY), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
pushed_refs = subprocess.run(
    ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if head != REVIEWED_COMMIT or dirty or not pushed_refs:
    raise RuntimeError(
        "Reproducibility gate failed: "
        f"head={head} dirty={bool(dirty)} pushed={bool(pushed_refs)}"
    )
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps",
        "-e", str(REPOSITORY),
    ],
    check=True,
)
SOURCE_ROOT = (REPOSITORY / "src").resolve()
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + (
    os.pathsep + existing_pythonpath if existing_pythonpath else ""
)
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != SOURCE_ROOT]
sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
for module_name in [
    name for name in sys.modules if name == "commguard" or name.startswith("commguard.")
]:
    del sys.modules[module_name]
import commguard

commguard_path = Path(commguard.__file__).resolve()
try:
    commguard_path.relative_to(SOURCE_ROOT)
except ValueError as exc:
    raise RuntimeError(f"CommGuard imported outside reviewed source: {commguard_path}") from exc
print({
    "reviewed_commit": head,
    "remote_refs": pushed_refs.splitlines(),
    "commguard_import": str(commguard_path.relative_to(REPOSITORY)),
    "torchrun_pythonpath_prefix": os.environ["PYTHONPATH"].split(os.pathsep)[0],
})


In [ ]:
from commguard.artifacts import restore_archive, sha256_file

INPUT_ARCHIVE = Path("/kaggle/input/commguard-calibration-v3/commguard-calibration-v3-REPLACE.tar.gz")
EXPECTED_INPUT_SHA256 = ""  # Required: SHA-256 printed by the preceding notebook.
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_INPUT_SHA256):
    raise RuntimeError("Set EXPECTED_INPUT_SHA256 to the exact 64-character archive hash.")
actual_input_sha256 = sha256_file(INPUT_ARCHIVE)
if actual_input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        "Input archive hash mismatch: "
        f"expected={EXPECTED_INPUT_SHA256} actual={actual_input_sha256}"
    )
restore_archive(INPUT_ARCHIVE, ARTIFACTS, expected_sha256=EXPECTED_INPUT_SHA256)
print({"restored_archive": str(INPUT_ARCHIVE), "sha256": actual_input_sha256})


In [ ]:
from datetime import datetime, timezone

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")


from commguard.environment.preflight import check_environment, summarize_environment
from commguard.provenance import ProvenanceContext

CONTEXT = ProvenanceContext.create(
    corpus_id=f"corpus-benign-v2-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-benign-v2-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-benign-v2-{NOTEBOOK_RUN_ID}",
    notebook_version="commguard_benign_corpus_v2",
    input_archive_sha256=EXPECTED_INPUT_SHA256,
    random_seed=20260730,
    repository_root=REPOSITORY,
)
if CONTEXT.source_dirty or CONTEXT.source_commit != REVIEWED_COMMIT:
    raise RuntimeError("SDK provenance no longer matches the clean reviewed source commit.")
ENVIRONMENT = check_environment(strict=True, output=ARTIFACTS, provenance=CONTEXT)
print(summarize_environment(ENVIRONMENT))
print({
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "source_commit": CONTEXT.source_commit,
    "input_archive_sha256": CONTEXT.input_archive_sha256,
})


In [ ]:
from commguard.artifacts import sha256_file
from commguard.calibration import build_calibration_reference, verify_calibration_reference

PRIOR_CALIBRATION_ARTIFACT_PATH = Path("results/calibration-REPLACE.json")
EXPECTED_PRIOR_CALIBRATION_SHA256 = ""  # Copy from calibration_v3 output.
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_PRIOR_CALIBRATION_SHA256):
    raise RuntimeError("Set the exact prior calibration artifact SHA-256.")
if PRIOR_CALIBRATION_ARTIFACT_PATH.is_absolute() or ".." in PRIOR_CALIBRATION_ARTIFACT_PATH.parts:
    raise RuntimeError("Prior calibration path must be artifact-root-relative.")
prior_path = ARTIFACTS / PRIOR_CALIBRATION_ARTIFACT_PATH
if not prior_path.is_file():
    raise RuntimeError(f"Exact prior calibration is missing: {PRIOR_CALIBRATION_ARTIFACT_PATH}")
if sha256_file(prior_path) != EXPECTED_PRIOR_CALIBRATION_SHA256:
    raise RuntimeError("Exact prior calibration artifact hash does not match.")
PRIOR_CALIBRATION_REFERENCE = build_calibration_reference(
    ARTIFACTS,
    PRIOR_CALIBRATION_ARTIFACT_PATH,
    current_experiment_session_id=CONTEXT.experiment_session_id,
)
if PRIOR_CALIBRATION_REFERENCE["calibration_relationship"] != "prior_session":
    raise RuntimeError("Restored calibration must be labeled prior-session input evidence.")
_, PRIOR_CALIBRATION = verify_calibration_reference(
    ARTIFACTS,
    PRIOR_CALIBRATION_REFERENCE,
    require_current_session=False,
    require_supported=False,
)
print({
    "prior_session_calibration_reference": PRIOR_CALIBRATION_REFERENCE,
    "prior_status_under_its_saved_contract": PRIOR_CALIBRATION["status"],
    "used_as_current_collection_gate": False,
})


In [ ]:
from commguard.orchestrator import estimate_matrix, run_matrix

RUN_BENIGN_PILOT = True
RUN_STANDARD_BENIGN_MATRIX = False
RUN_EXPANDED_BENIGN_MATRIX = False

requested = []
if RUN_BENIGN_PILOT:
    requested.append(("smoke", 1))
if RUN_STANDARD_BENIGN_MATRIX:
    requested.append(("standard", 3))
if RUN_EXPANDED_BENIGN_MATRIX:
    requested.append(("extended", 3))
if len(requested) != 1:
    raise RuntimeError("Enable exactly one benign profile per immutable notebook archive.")
PROFILE, REPETITIONS = requested[0]
print({"estimate": estimate_matrix(PROFILE, REPETITIONS), "duration_aware": True})
MATRIX = run_matrix(
    PROFILE,
    output=ARTIFACTS,
    repetitions=REPETITIONS,
    timeout_s=180.0,
    provenance=CONTEXT,
    prior_calibration_reference=PRIOR_CALIBRATION_REFERENCE,
)
CURRENT_CALIBRATION_REFERENCE = MATRIX["calibration_reference"]
if CURRENT_CALIBRATION_REFERENCE["calibration_relationship"] != "current_session":
    raise RuntimeError("Fresh collection calibration was not labeled current_session.")
if CURRENT_CALIBRATION_REFERENCE == PRIOR_CALIBRATION_REFERENCE:
    raise RuntimeError("Prior and current calibration references must not be interchangeable.")
print({
    "prior_session_input_calibration": PRIOR_CALIBRATION_REFERENCE,
    "current_session_gating_calibration": CURRENT_CALIBRATION_REFERENCE,
    "benign_matrix_summary_path_for_next_notebook": MATRIX["summary_artifact"],
    "feature_extraction_summary_for_next_notebook": MATRIX["feature_extraction_summary"],
})


In [ ]:
COVERAGE_TABLE = [
    {"family": family, **counts}
    for family, counts in sorted(MATRIX["family_counts"].items())
]
for row in COVERAGE_TABLE:
    print(row)
print({
    "primary_coverage_gate": MATRIX["primary_coverage_gate"],
    "detector_metrics_computed": MATRIX["detector_metrics_computed"],
})


## Results

not executed. Coverage and completion are unknown until the notebook is run.


In [ ]:
from commguard.artifacts import ArtifactStore, sha256_file

ARCHIVE = Path(f"/kaggle/working/commguard-benign-corpus-v2-{NOTEBOOK_RUN_ID}.tar.gz")
ArtifactStore(ARTIFACTS).export(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)
SHA_FILE = ARCHIVE.with_suffix(ARCHIVE.suffix + ".sha256")
SHA_FILE.write_text(f"{ARCHIVE_SHA256}  {ARCHIVE.name}\n", encoding="utf-8")
print(f"NEXT STEP: add {ARCHIVE} to a private Kaggle dataset without renaming it.")
print(f"NEXT STEP: copy SHA-256 {ARCHIVE_SHA256} into EXPECTED_INPUT_SHA256 in commguard_detector_evaluation_v2.ipynb.")
print(
    f"NEXT STEP: set that notebook's REVIEWED_COMMIT to {REVIEWED_COMMIT} "
    "and run from the first cell."
)
